<a href="https://colab.research.google.com/github/KyryloKozlovskyi/garbage-classification/blob/main/garbage_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Garbage Image Classification

**Module:** Machine Learning Technologies  
**Student ID:** G00425385  
**Dataset:** Garbage Classification - 10 categories, 384 × 384 RGB  
**Random Seed:** 425385  

---
## 1. Environment Setup

All dependencies are imported upfront so the notebook can be restarted and re-run cleanly.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os, time, timeit, zipfile

from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import VGG16
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 425385
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow: {tf.__version__}')
gpu_devices = tf.config.list_physical_devices('GPU')
print(f'GPU detected: {bool(gpu_devices)}  {gpu_devices}')

## 2. Dataset Preparation

The dataset zip is stored on Google Drive. It is extracted to Colab's local disk for faster I/O during training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Path configuration ----
ZIP_FILE = '/content/drive/MyDrive/Garbage_data.zip'   # <- update if needed
LOCAL_DIR = '/content/garbage_dataset'

if not os.path.exists(ZIP_FILE):
    print(f'ERROR: {ZIP_FILE} not found.')
    print('Contents of Drive root:')
    for f in sorted(os.listdir('/content/drive/MyDrive'))[:15]:
        print(f'  {f}')
else:
    mb = os.path.getsize(ZIP_FILE) / 1024**2
    print(f'Zip found: {ZIP_FILE} ({mb:.0f} MB)')

In [ ]:
# Extract once to local disk
if not os.path.exists(LOCAL_DIR):
    print('Extracting ...')
    with zipfile.ZipFile(ZIP_FILE, 'r') as zf:
        zf.extractall(LOCAL_DIR)
    print('Extraction complete.')
else:
    print('Already extracted.')

In [ ]:
# Locate the folder that holds the 10 class sub-directories
TARGET_CLASSES = {
    'battery', 'biological', 'cardboard', 'clothes', 'glass',
    'metal', 'paper', 'plastic', 'shoes', 'trash'
}

DATA_ROOT = None
for dirpath, dirnames, _ in os.walk(LOCAL_DIR):
    if TARGET_CLASSES.issubset(set(dirnames)):
        DATA_ROOT = dirpath
        break

if DATA_ROOT:
    print(f'Dataset root: {DATA_ROOT}')
else:
    print('Could not auto-detect — set DATA_ROOT manually below.')

In [ ]:
# Uncomment and set manually only if auto-detection failed
# DATA_ROOT = '/content/garbage_dataset/Garbage_data/Garbage_data'

---
## 3. Exploratory Data Analysis

### 3.1 Class Distribution

Before any modelling, inspecting how many images belong to each category reveals whether class imbalance is present. If one class dominates, accuracy becomes misleading because a model that always predicts the majority class can still score well.

In [ ]:
# Count files in each class folder
categories = sorted(
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
)

img_counts = {}
for cat in categories:
    folder = os.path.join(DATA_ROOT, cat)
    n = len([f for f in os.listdir(folder) if not f.startswith('.')])
    img_counts[cat] = n

total = sum(img_counts.values())
largest = max(img_counts, key=img_counts.get)
smallest = min(img_counts, key=img_counts.get)

print(f'{"Category":<14} {"Count":>6}')
print('-' * 22)
for cat, n in sorted(img_counts.items(), key=lambda x: -x[1]):
    print(f'{cat:<14} {n:>6}')
print('-' * 22)
print(f'{"Total":<14} {total:>6}')
print()
print(f'Largest:  {largest} ({img_counts[largest]})')
print(f'Smallest: {smallest} ({img_counts[smallest]})')
print(f'Ratio:    {img_counts[largest] / img_counts[smallest]:.1f}x')

In [ ]:
# Visualise the distribution
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(img_counts.keys(), img_counts.values(),
              color=plt.cm.tab10.colors[:len(img_counts)])

for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 15,
            int(b.get_height()), ha='center', fontsize=8)

ax.set_ylabel('Number of images')
ax.set_title('Images per Category')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

### 3.2 Sample Images

Viewing a few images from every class gives an intuition for how varied the data is and which categories might be visually similar.

In [ ]:
n_show = 3  # images per class
fig, axes = plt.subplots(n_show, len(categories), figsize=(22, 7))

for col_idx, cat in enumerate(categories):
    folder = os.path.join(DATA_ROOT, cat)
    samples = sorted(os.listdir(folder))[:n_show]
    for row_idx, fname in enumerate(samples):
        img = tf.keras.utils.load_img(
            os.path.join(folder, fname), target_size=(128, 128)
        )
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis('off')
    axes[0, col_idx].set_title(cat, fontsize=9)

fig.suptitle('Sample Images per Category', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Image Dimensions and Pixel Values

Confirming that images have consistent dimensions and checking the raw pixel range.

In [ ]:
# Sample a few images per class to check dimensions
shapes_seen = set()
px_lo, px_hi = 255, 0

for cat in categories:
    folder = os.path.join(DATA_ROOT, cat)
    for fname in sorted(os.listdir(folder))[:5]:
        arr = np.array(
            tf.keras.utils.load_img(os.path.join(folder, fname))
        )
        shapes_seen.add(arr.shape)
        px_lo = min(px_lo, arr.min())
        px_hi = max(px_hi, arr.max())

print(f'Unique shapes found: {shapes_seen}')
print(f'Pixel range: [{px_lo}, {px_hi}]')

---
## 4. Data Loading and Splitting

The data is loaded with `image_dataset_from_directory` and split 70 / 10 / 20 (train / validation / test). Keras only supports a single split parameter, so a 70/30 split is performed first, then the 30 % holdout is divided into one-third validation and two-thirds test.

The **validation** set is used throughout to compare models and select hyperparameters. The **test** set is reserved for a single final evaluation of the chosen model.

In [ ]:
# ---- Hyperparameters ----
IMG_H = 128
IMG_W = 128
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# 70 % training
ds_train = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.3,
    subset='training',
    seed=SEED,
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH,
    label_mode='int'
)

# 30 % holdout
ds_holdout = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.3,
    subset='validation',
    seed=SEED,
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH,
    label_mode='int'
)

# Split holdout: first third → validation, remainder → test
holdout_size = tf.data.experimental.cardinality(ds_holdout).numpy()
n_val = holdout_size // 3

ds_val = ds_holdout.take(n_val)
ds_test = ds_holdout.skip(n_val)

label_names = ds_train.class_names
n_classes = len(label_names)

print(f'Labels ({n_classes}): {label_names}')
print(f'Batches  — train: {tf.data.experimental.cardinality(ds_train).numpy()}, '
      f'val: {tf.data.experimental.cardinality(ds_val).numpy()}, '
      f'test: {tf.data.experimental.cardinality(ds_test).numpy()}')

In [ ]:
# Verify split sizes
n_train = sum(labels.shape[0] for _, labels in ds_train)
n_val_img = sum(labels.shape[0] for _, labels in ds_val)
n_test = sum(labels.shape[0] for _, labels in ds_test)

print(f'Images   — train: {n_train}, val: {n_val_img}, test: {n_test}')
print(f'Total: {n_train + n_val_img + n_test}')

In [ ]:
# Quick sanity check: shape and range of one batch
for batch_imgs, batch_labels in ds_train.take(1):
    print(f'Image batch: {batch_imgs.shape}  dtype={batch_imgs.dtype}')
    print(f'Label batch: {batch_labels.shape}  dtype={batch_labels.dtype}')
    print(f'Pixel range: [{batch_imgs.numpy().min():.0f}, {batch_imgs.numpy().max():.0f}]')

### 4.1 Class Weights

Because the dataset is imbalanced, class weights are computed so that misclassifying a rare category (e.g. trash) costs more than misclassifying a common one (e.g. clothes). These weights are passed to `model.fit()` through the `class_weight` parameter.

In [ ]:
# Gather all training labels
all_train_labels = np.concatenate(
    [labels.numpy() for _, labels in ds_train]
)

cw = compute_class_weight(
    'balanced',
    classes=np.arange(n_classes),
    y=all_train_labels
)
class_weights = {i: w for i, w in enumerate(cw)}

print(f'{"Class":<14} {"Weight":>8}')
print('-' * 24)
for i, name in enumerate(label_names):
    print(f'{name:<14} {class_weights[i]:>8.4f}')

### 4.2 Performance Optimisation

Prefetching allows the data pipeline to prepare the next batch while the GPU is training on the current one.

In [ ]:
ds_train = ds_train.cache().prefetch(AUTOTUNE)
ds_val   = ds_val.cache().prefetch(AUTOTUNE)
ds_test  = ds_test.cache().prefetch(AUTOTUNE)

---
## 5. Utility Functions

Reusable helpers for plotting training curves, computing metrics, and displaying confusion matrices.

In [ ]:
def show_curves(hist, title=''):
    """Side-by-side accuracy and loss plots from a Keras History object."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    if title:
        fig.suptitle(title, fontsize=13)

    a1.plot(hist.history['accuracy'],     label='train')
    a1.plot(hist.history['val_accuracy'],  label='val')
    a1.set_xlabel('Epoch'); a1.set_ylabel('Accuracy')
    a1.legend(); a1.grid(True, alpha=0.3)

    a2.plot(hist.history['loss'],     label='train')
    a2.plot(hist.history['val_loss'],  label='val')
    a2.set_xlabel('Epoch'); a2.set_ylabel('Loss')
    a2.legend(); a2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def get_predictions(model, dataset):
    """Return ground-truth and predicted label arrays for a dataset."""
    y_true, y_pred = [], []
    for imgs, lbls in dataset:
        logits = model.predict(imgs, verbose=0)
        y_true.append(lbls.numpy())
        y_pred.append(np.argmax(logits, axis=1))
    return np.concatenate(y_true), np.concatenate(y_pred)


def report_metrics(model, dataset, tag='Validation'):
    """Print accuracy, macro-F1, per-class report, and return predictions."""
    y_true, y_pred = get_predictions(model, dataset)

    acc  = np.mean(y_true == y_pred)
    mf1  = f1_score(y_true, y_pred, average='macro')

    print(f'\n=== {tag} ===')
    print(f'Accuracy :  {acc:.4f}')
    print(f'Macro F1 :  {mf1:.4f}')
    print(f'Parameters: {model.count_params():,}')
    print()
    print(classification_report(y_true, y_pred,
                                target_names=label_names))
    return y_true, y_pred, mf1


def show_confusion(y_true, y_pred, title=''):
    """Display a labelled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
    fig, ax = plt.subplots(figsize=(9, 7))
    disp.plot(ax=ax, cmap='Oranges', xticks_rotation=45)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def measure_inference(model, dataset, n_runs=10):
    """Average inference time over n_runs using timeit on one batch."""
    one_batch = next(iter(dataset))[0]
    t = timeit.timeit(
        lambda: model.predict(one_batch, verbose=0),
        number=n_runs
    ) / n_runs
    print(f'Mean inference time (1 batch, {n_runs} runs): {t:.4f} s')
    return t

---

**Phase 1 complete.** The dataset is loaded, explored, split, and the helper functions are ready.

Next: **Phase 2 — CNN from Scratch (RGB)**